### Book store sales Dataset 4

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 

df = pd.read_csv("train.csv", low_memory = False)
df.head()
df["Date"] = pd.to_datetime(df["Date"])

store_1 = df[df["Store"] == 1] 

ts = store_1.set_index("Date")["Sales"]
ts = ts.sort_index()

print(ts.head())
print(ts.tail())
print(len(ts))

In [ ]:
plt.figure()
plt.plot(ts.index, ts.values)
plt.xlabel("Date")
plt.ylabel("Sales")
plt.title("Sales vs Date")
plt.show()

mean = np.mean(ts.values)
variance = np.var(ts.values)

print(mean)
print(variance)

In [ ]:
from statsmodels.tsa.stattools import acf

acf_values = acf(signal, nlags = 7)

lags = range(len(acf_values))

critical_value = 1.96 / np.sqrt(len(ts))

plt.figure()
plt.stem(lags, acf_values)
plt.xlabel("Date")
plt.axhline(critical_value,linestyle='--')
plt.axhline(-critical_value,linestyle='--')
plt.ylabel("ACF Sales")
plt.title("ACF Sales vs Date")
plt.show()

#### Strong autocorrelation as day 7 - seasonal likely, lets use Dicky fueller 

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(ts.values)

print("ADF Statistic:" , result[0])
print("p-value:", result[1])

#### ADF test indicates that stationarity looks fine. Big spike at lag 7 in the ACF. We'll need to do SARIMA 

#### Starting again from the beginning

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf

df = pd.read_csv("train.csv")

df.info

df["Date"] = pd.to_datetime(df["Date"])

store_1 = df[df["Store"] == 1]

print(len(store_1))


In [ ]:
ts = store_1.set_index("Date")[["Sales"]]
ts = ts.sort_index()

data = ts.loc["2013-01-01":"2013-12-31"]   

plt.figure(figsize = (16,4))
plt.plot(data)
plt.xlabel("Date")
plt.xticks(data.index[::30], rotation=45)
plt.ylabel("Sales")
plt.title("Sales vs Date")
plt.show()

I can visually see the seasonality here. Once a week, likely on weekends. Let's confirm the actual dates of which we are seeing the spikes. We can also observe what looks like a yearly seasonality, where the sales increase at the end of the year likely due to christmas shoppers. We'll check both yearly and weekly seasonality. 

In [ ]:
type(data)

data.index.day_name()

data["Day"] = data.index.day_name()

pd.set_option('display.max_rows', None)
print(data.columns)


To best see a possible weekly seasonality it makes more sense to graph it and use a smaller range. Let's do 2 months 

In [ ]:
data = ts.loc["2013-01-01":"2013-03-01"]   

plt.figure(figsize = (16,4))
plt.plot(data)
plt.xlabel("Date")
plt.xticks(data.index[::7], rotation=45)
plt.ylabel("Sales")
plt.title("Sales vs Date")
plt.show()

One thing I realized is that theres no sales at all on Sunday. The store must be closed. However, it can still be observed that the data has a cyclic pattern by week even with the drop on sunday. Out of curiousity, Let's check the sales by day of the week. 

In [ ]:
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

data["Day"] = data.index.day_name()

data["Day"] = pd.Categorical(
    data["Day"],
    categories=order,
    ordered=True
)
data.boxplot(column="Sales", by="Day")
plt.title("Sales by Day of Week")
plt.suptitle("")
plt.show()

In [ ]:
print(data.groupby("Day")["Sales"].median())

So it looks like the most sales happen on Wednesdays and saturdays, showing lower values on the days between. There is definitely weekly seasonality. Let's take a look at the PACF and the ACF.

In [ ]:
data = ts.loc["2008-01-01":"2015-01-01"]   

plt.figure(figsize = (16,4))
plt.plot(data)
plt.xlabel("Date")
plt.xticks(data.index[::30], rotation=45)
plt.ylabel("Sales")
plt.title("Sales vs Date")
plt.show()

There's definitely yearly seasonality, but since im not sure how to deal with that with the weekly as well, we'll just focus on the clear weekly seasonality. Let's do the PACF and ACF 

In [ ]:
plt.figure()
plot_acf(data, lags = 30 )
plt.show()

plt.figure()
plot_pacf(data, lags= 30)

Seasonality is confirmed by the PACF and the ACF. Let's implement SARIMA for the first two months of 2013. We'll do a forecast and see how it is compared to the real data. 

In [ ]:
from pmdarima import auto_arima  

data = ts.loc["2013-01-01":"2013-02-01"]  

model = auto_arima(data, seasonal = True, trace = True, stepwise=True, m = 7)

print(model.summary())

In [ ]:
residuals = model.resid()

plt.figure()
plt.plot(residuals)
plt.xlabel("Date")
plt.xticks(residuals.index[::7], rotation=45)
plt.ylabel("Residuals")

In [ ]:
residuals_trimmed = residuals[14:]

plt.figure()
plt.plot(residuals_trimmed)
plt.xlabel("Date")
plt.xticks(residuals.index[14::7], rotation=45)
plt.ylabel("Residuals")

In [ ]:
plt.figure()
plot_acf(residuals_trimmed)

plt.figure()
plot_pacf(residuals_trimmed)

In [ ]:
from diagnostics import diagnostic_plot

diagnostic_plot(residuals_trimmed)

Plots look pretty good, but I fed a relatively small dataset here. I'm wondering if i can improve it by increasing the size of that. Let's re do this, with a larger splice, but first let's do a forecast and compare our error to the actual. 


In [ ]:
fitted_values = model.predict_in_sample()

plt.figure()
plt.plot(data, label="Actual")
plt.plot(fitted_values, label="Fitted")
plt.legend()
plt.title("ARIMA Fit vs Actual")
plt.xticks(residuals.index[14::7], rotation=45)
plt.show() 

I can see the forecast looks a bit rough. Seems to get better as time goes on. Let's try to fix this by increasing the data splice.

In [ ]:
data = ts.loc["2013-01-01":"2013-06-01"]  

model = auto_arima(data, seasonal = True, trace = True, stepwise=True, m = 7)

print(model.summary())

In [ ]:
residuals = model.resid()

residuals_trimmed = residuals[14:]

plt.figure()
plt.plot(residuals_trimmed)
plt.xlabel("Date")
plt.xticks(residuals.index[14::7], rotation=45)
plt.ylabel("Residuals")

In [ ]:
plt.figure()
plot_acf(residuals_trimmed)

plt.figure()
plot_pacf(residuals_trimmed)

Increasing the splice looks like it made the PACF and ACF worse. I think this is because theres more to the data other than the weekly seasonality, and it's picking that up. Just going to check the diagnostics.


In [ ]:
diagnostic_plot(residuals_trimmed)

In [ ]:
df.info

In [ ]:
plt.figure()
plt.plot(data)


Let's try to use VAR to improve the model with using sales and customers. First creating the two columns from store 1 that we'll use. 

In [ ]:
from statsmodels.tsa.api import VAR

ts = store_1.set_index("Date")[["Sales","Customers"]].dropna()
ts = ts.sort_index()
 

In [ ]:
ts.index

Let's now difference the data at the known 7 day lag we found before to remove the seasonality

In [ ]:
ts_diff = ts.diff(7).dropna()

In [ ]:
plot_acf(ts_diff["Customers"],title= "Autocorrelation Customers")
plot_acf(ts_diff["Sales"],title= "Autocorrelation Sales")

In [ ]:
print(ts.shape)

In [ ]:
ts.describe

In [ ]:
ts_diff.head(10)

We still have some seasonality left there based on the PACF and ACF graphs. I notice it's still at 7 14 21 28 ish day lags. Let's try removing the trend by y(t) - y(t-1) and y(t) - y(t-7)

In [ ]:
ts_diff2 = ts.diff().diff(7).dropna()


In [ ]:
plot_acf(ts_diff2["Sales"],title = "ACF for Sales")
plot_acf(ts_diff2["Customers"],title = "ACF for Customers")
plot_pacf(ts_diff2["Sales"],title = "PACF for Sales")
plot_pacf(ts_diff2["Customers"],title = "PACF for Customers")

Looks like we removed most of the seasonality. Let's try VAR 

In [ ]:
model = VAR(ts_diff2)

results = model.fit(maxlags=10, ic='aic')

print(results.summary())

Interesting. We see the p values indicate that customers at many of the time steps influence the sales because the p value is below 0.05 and also the coefficient is very large for customers! we can also see that the relation between sales and itself is much lower! very interesting. 

In [ ]:
residuals = results.resid

In [ ]:
plot_acf(residuals["Sales"], title = "Residuals for Customers")
plot_acf(residuals["Customers"], title = "Residuals for Sales")

In [ ]:
diagnostic_plot(residuals["Sales"])

In [ ]:
residuals.shape

I forgot to set the correct time splice, accidentally used the entire data set. Let's use the same 5 months as the previous ARIMA to compare the results. 

In [ ]:
data = ts.loc["2013-01-01":"2013-06-01"] 

In [ ]:
data_diff = data.diff().diff(7).dropna()

In [ ]:
plot_acf(data_diff["Customers"],title= "Autocorrelation Customers")
plot_acf(data_diff["Sales"],title= "Autocorrelation Sales")

Spike goes up to lag 8 ish, lets choose 10 to make sure we capture it 

In [ ]:
model = VAR(data_diff)

results = model.fit(maxlags=10, ic='aic')

print(results.summary())

In [ ]:
diagnostic_plot(results.resid["Sales"])

I want to do a forecast with both the VAR model and the ARIMA model and check the error to see which one is more accurate. 

In [ ]:
data = ts.loc["2013-01-01":"2013-06-01"]   

plt.figure(figsize = (16,4))
plt.plot(data)
plt.xlabel("Date")
plt.xticks(data.index[::30], rotation=45)
plt.ylabel("Sales")
plt.title("Sales vs Date")
plt.show()